# 📊 Sesión 13 — Comparación y Selección de Modelos de Credit Scoring

**Curso:** Modelos de Credit Scoring con Machine Learning (Parte 2)  
**Objetivo:** Comparar los 4 modelos candidatos (+ Logistic Regression como baseline), seleccionar el modelo final y justificar la decisión en términos de riesgo de crédito.

---

## 📌 Agenda

| # | Sección | Descripción |
|---|---------|-------------|
| 0 | Setup | Librerías, datos, carga de modelos |
| 1 | Baseline | Logistic Regression como punto de referencia |
| 2 | Discriminación | AUC-ROC, Gini, KS, Curva PR |
| 3 | Calibración | Brier Score, Hosmer-Lemeshow, Reliability Diagram |
| 4 | Análisis Económico | Curvas de Ganancia, Lift, Cost-Benefit |
| 5 | Estabilidad | CV estratificada, PSI, Overfitting |
| 6 | Dashboard | Tabla consolidada + Radar Chart |
| 7 | Selección Final | Score ponderado + justificación regulatoria |
| 8 | Conclusiones | Recomendaciones y próximos pasos |

---

> **Referencias clave:**  
> - Siddiqi, N. (2006). *Credit Risk Scorecards*. Wiley.  
> - Thomas, L.C., Edelman, D.B., & Crook, J.N. (2002). *Credit Scoring and Its Applications*. SIAM.  
> - Baesens, B. & Van Gestel, T. (2009). *Credit Risk Management*. Oxford University Press.  
> - Lessmann, S. et al. (2015). Benchmarking state-of-the-art classification algorithms for credit scoring. *EJOR, 247*(1), 124-136.  
> - BIS BCBS (2005). *Studies on the Validation of Internal Rating Systems*. Working Paper No. 14.  
> - EBA (2017). *Guidelines on PD estimation, LGD estimation and defaulted exposures*. EBA/GL/2017/16.

---
## ⚙️ Sección 0 — Setup: Librerías, Datos y Carga de Modelos

In [ ]:
# ── Librerías ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    brier_score_loss, confusion_matrix, log_loss
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import StratifiedKFold, cross_validate
from scipy.stats import chi2
import joblib

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800']
# Azul=LR Baseline | Naranja=DT | Verde=RF | Púrpura=XGB | Ámbar=LGBM

print("✅ Librerías cargadas correctamente")

In [ ]:
# ── Carga de Datos ───────────────────────────────────────────────────────────
# Ajusta rutas según la estructura de tu repositorio
# Opción A: CSV
X_train = pd.read_csv('../data/X_train.csv')
X_test  = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/y_test.csv').squeeze()

# Opción B: pickle
# import pickle
# with open('../data/splits.pkl', 'rb') as f:
#     X_train, X_test, y_train, y_test = pickle.load(f)

print(f"Train : {X_train.shape} | Test : {X_test.shape}")
print(f"Default Rate (Train): {y_train.mean():.2%}")
print(f"Default Rate (Test) : {y_test.mean():.2%}")

In [ ]:
# ── Carga de Modelos Pre-entrenados (Sesiones 11 y 12) ───────────────────────
model_dt  = joblib.load('../models/decision_tree.pkl')
model_rf  = joblib.load('../models/random_forest.pkl')
model_xgb = joblib.load('../models/xgboost.pkl')
model_lgb = joblib.load('../models/lightgbm.pkl')

print("✅ Modelos cargados: Decision Tree, Random Forest, XGBoost, LightGBM")

---
## 📏 Sección 1 — Baseline: Logistic Regression

La Regresión Logística es el modelo estándar de la industria bancaria para credit scoring
(Siddiqi, 2006; Thomas et al., 2002). Usarla como **baseline** permite responder:

> *¿Cuánta ganancia real aportan los modelos de ML respecto al modelo regulatorio estándar?*

Esto es clave para justificar complejidad adicional ante un supervisor (BIS BCBS, 2005).

In [ ]:
# ── Entrenamiento del Baseline ────────────────────────────────────────────────
model_lr = LogisticRegression(
    penalty='l2',
    C=1.0,
    solver='lbfgs',
    max_iter=1000,
    class_weight='balanced',   # importante con clases desbalanceadas
    random_state=42
)
model_lr.fit(X_train, y_train)

# Resumen de coeficientes
coef_df = pd.DataFrame({
    'Variable'    : X_train.columns,
    'Coeficiente' : model_lr.coef_[0],
    'Odds Ratio'  : np.exp(model_lr.coef_[0])
}).sort_values('Coeficiente', ascending=False)

print("\n📋 Top 10 Variables por Magnitud de Coeficiente:")
print(coef_df.head(10).to_string(index=False))

In [ ]:
# ── Gráfica de Coeficientes (Top 15) ─────────────────────────────────────────
top_vars = coef_df.reindex(coef_df['Coeficiente'].abs().nlargest(15).index)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#FF5722' if c > 0 else '#2196F3' for c in top_vars['Coeficiente']]
ax.barh(top_vars['Variable'], top_vars['Coeficiente'], color=colors)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Logistic Regression — Coeficientes (Baseline)', fontsize=13, fontweight='bold')
ax.set_xlabel('Coeficiente')
red_patch  = mpatches.Patch(color='#FF5722', label='Aumenta PD')
blue_patch = mpatches.Patch(color='#2196F3', label='Reduce PD')
ax.legend(handles=[red_patch, blue_patch])
plt.tight_layout()
plt.savefig('../outputs/sesion13_lr_coeficientes.png', dpi=150)
plt.show()

---
## 📐 Sección 2 — Métricas de Discriminación

Las métricas de discriminación evalúan **qué tan bien el modelo separa buenos de malos pagadores**.
Son las más utilizadas en la práctica bancaria y en guías regulatorias (EBA, 2017; BIS BCBS, 2005).

| Métrica | Interpretación en Credit Scoring | Umbral Referencial |
|---------|----------------------------------|--------------------|
| **AUC-ROC** | Prob. de que un malo score > que un bueno | > 0.75 aceptable, > 0.80 bueno |
| **Gini** | = 2·AUC - 1; el más usado en scoring bancario | > 0.50 aceptable |
| **KS** | Máxima separación acumulada entre buenos y malos | > 0.40 aceptable |
| **PR-AUC** | AUC de curva Precisión-Recall; mejor con desbalance | Depende del contexto |

> **Referencia:** Siddiqi (2006, Cap. 10); Lessmann et al. (2015, EJOR 247(1)).

In [ ]:
# ── Funciones Auxiliares ─────────────────────────────────────────────────────
def compute_ks(y_true, y_proba):
    """Kolmogorov-Smirnov: máxima diferencia entre CDF de buenos y malos."""
    df = pd.DataFrame({'y': y_true, 'p': y_proba})
    df = df.sort_values('p', ascending=False).reset_index(drop=True)
    n_bad  = df['y'].sum()
    n_good = len(df) - n_bad
    df['cum_bad']  = df['y'].cumsum() / n_bad
    df['cum_good'] = (1 - df['y']).cumsum() / n_good
    df['ks']       = (df['cum_bad'] - df['cum_good']).abs()
    return df['ks'].max()

def compute_gini(y_true, y_proba):
    """Gini = 2 * AUC - 1."""
    return 2 * roc_auc_score(y_true, y_proba) - 1

# Diccionario unificado de modelos
MODELS = {
    'Logistic Reg (Baseline)': (model_lr,  PALETTE[0]),
    'Decision Tree'           : (model_dt,  PALETTE[1]),
    'Random Forest'           : (model_rf,  PALETTE[2]),
    'XGBoost'                 : (model_xgb, PALETTE[3]),
    'LightGBM'                : (model_lgb, PALETTE[4]),
}

PROBAS = {
    name: model.predict_proba(X_test)[:, 1]
    for name, (model, _) in MODELS.items()
}
print("✅ Funciones y diccionario de modelos listos")

In [ ]:
# ── Tabla de Métricas de Discriminación ──────────────────────────────────────
rows = []
for name, proba in PROBAS.items():
    rows.append({
        'Modelo'  : name,
        'AUC-ROC' : round(roc_auc_score(y_test, proba), 4),
        'Gini'    : round(compute_gini(y_test, proba), 4),
        'KS'      : round(compute_ks(y_test, proba), 4),
        'PR-AUC'  : round(average_precision_score(y_test, proba), 4),
        'Log-Loss': round(log_loss(y_test, proba), 4),
    })

metrics_df = pd.DataFrame(rows).set_index('Modelo')
print("\n📊 Métricas de Discriminación — Conjunto de Test\n")
print(metrics_df.to_string())

In [ ]:
# ── Curvas ROC ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

for (name, (_, color)), proba in zip(MODELS.items(), PROBAS.values()):
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc_val = auc(fpr, tpr)
    lw = 1.8 if 'Baseline' in name else 2.5
    ls = '--'  if 'Baseline' in name else '-'
    ax.plot(fpr, tpr, color=color, lw=lw, ls=ls,
            label=f'{name}  (AUC={roc_auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=0.8, label='Random (AUC=0.500)')
ax.set_xlabel('Tasa de Falsos Positivos (1 - Especificidad)', fontsize=11)
ax.set_ylabel('Tasa de Verdaderos Positivos (Sensibilidad)', fontsize=11)
ax.set_title('Curvas ROC — Comparación de Modelos', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig('../outputs/sesion13_roc_curves.png', dpi=150)
plt.show()

In [ ]:
# ── Tabla KS por Decil (modelo con mayor KS) ─────────────────────────────────
def ks_table_by_decile(y_true, y_proba, n_bins=10):
    df = pd.DataFrame({'y': y_true.values, 'p': y_proba})
    df['decile'] = pd.qcut(df['p'], n_bins, labels=False, duplicates='drop')
    df['decile'] = n_bins - df['decile']
    grp = df.groupby('decile').agg(n=('y','count'), n_bad=('y','sum')).reset_index()
    grp['n_good']      = grp['n'] - grp['n_bad']
    grp['cum_bad_pct'] = grp['n_bad'].cumsum()  / grp['n_bad'].sum()
    grp['cum_good_pct']= grp['n_good'].cumsum() / grp['n_good'].sum()
    grp['ks']          = (grp['cum_bad_pct'] - grp['cum_good_pct']).abs()
    grp['default_rate']= grp['n_bad'] / grp['n']
    return grp

best_ks_name = metrics_df['KS'].idxmax()
ks_df = ks_table_by_decile(y_test, PROBAS[best_ks_name])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(ks_df['decile'], ks_df['cum_bad_pct'],  'r-o', lw=2, label='% Acum. Malos')
ax.plot(ks_df['decile'], ks_df['cum_good_pct'], 'b-o', lw=2, label='% Acum. Buenos')
ax.fill_between(ks_df['decile'], ks_df['cum_bad_pct'], ks_df['cum_good_pct'],
                alpha=0.15, color='green', label=f'KS = {ks_df["ks"].max():.3f}')
ax.set_title(f'Curva KS — {best_ks_name}', fontweight='bold')
ax.set_xlabel('Decil (1=Mayor Riesgo)'); ax.set_ylabel('% Acumulado')
ax.legend()

ax2 = axes[1]
ax2.bar(ks_df['decile'], ks_df['default_rate'] * 100,
        color=plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(ks_df))))
ax2.axhline(y_test.mean() * 100, color='black', ls='--', lw=1.5,
            label=f'Promedio: {y_test.mean():.1%}')
ax2.set_title('Tasa de Default por Decil', fontweight='bold')
ax2.set_xlabel('Decil (1=Mayor Riesgo)'); ax2.set_ylabel('Default Rate (%)')
ax2.legend()

plt.tight_layout()
plt.savefig('../outputs/sesion13_ks_analysis.png', dpi=150)
plt.show()

print("\n📋 Tabla KS por Decil:")
print(ks_df[['decile','n','n_bad','default_rate','cum_bad_pct','cum_good_pct','ks']].to_string(index=False))

---
## 🎯 Sección 3 — Métricas de Calibración

La **calibración** mide si las probabilidades predichas corresponden a las tasas de default reales.
Para modelos bajo **IFRS 9 y Basilea**, un modelo bien calibrado es **condición necesaria**
para que las PD estimadas sean admisibles como insumo de provisiones (EBA, 2017).

> *"A model that discriminates well but is poorly calibrated is useless for capital computation."*  
> — Tasche (2006), *Validation of internal rating systems*

| Métrica | Qué mide | Ideal |
|---------|----------|-------|
| **Brier Score** | Error cuadrático medio de probabilidades | → 0 |
| **Hosmer-Lemeshow** | Chi² entre PD observada y esperada por decil | p-value > 0.05 |
| **Reliability Diagram** | Gráfica de calibración (observed vs predicted) | Diagonal perfecta |

In [ ]:
# ── Test de Hosmer-Lemeshow ───────────────────────────────────────────────────
def hosmer_lemeshow_test(y_true, y_proba, g=10):
    """
    Test de Hosmer-Lemeshow.
    Referencia: Hosmer & Lemeshow (2000), Applied Logistic Regression, 2nd ed.
    H0: el modelo está bien calibrado (p-value > 0.05).
    """
    df = pd.DataFrame({'y': y_true.values, 'p': y_proba})
    df['decile'] = pd.qcut(df['p'], g, duplicates='drop')
    grp = df.groupby('decile').agg(n=('y','count'), obs=('y','sum'), pred=('p','sum')).reset_index()
    pi = grp['pred'] / grp['n']
    pi = pi.clip(1e-6, 1 - 1e-6)
    hl_stat = ((grp['obs'] - grp['pred'])**2 / (grp['n'] * pi * (1 - pi))).sum()
    p_val = 1 - chi2.cdf(hl_stat, df=g - 2)
    return hl_stat, p_val

# ── Tabla de Calibración ──────────────────────────────────────────────────────
cal_rows = []
for name, proba in PROBAS.items():
    brier = brier_score_loss(y_test, proba)
    try:
        hl_stat, hl_p = hosmer_lemeshow_test(y_test, proba)
        calibrado = '✅' if hl_p > 0.05 else '⚠️'
    except Exception:
        hl_stat, hl_p, calibrado = np.nan, np.nan, 'N/A'
    cal_rows.append({
        'Modelo'        : name,
        'Brier Score'   : round(brier, 5),
        'HL Statistic'  : round(hl_stat, 3) if not np.isnan(hl_stat) else 'N/A',
        'HL p-value'    : round(hl_p, 4)    if not np.isnan(hl_p)    else 'N/A',
        'Bien Calibrado': calibrado
    })

cal_df = pd.DataFrame(cal_rows).set_index('Modelo')
print("\n🎯 Métricas de Calibración:\n")
print(cal_df.to_string())

In [ ]:
# ── Reliability Diagram + Distribución de Scores ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Calibración perfecta')
for (name, (_, color)), proba in zip(MODELS.items(), PROBAS.values()):
    frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=10)
    lw = 1.8 if 'Baseline' in name else 2.5
    ax.plot(mean_pred, frac_pos, 's-', color=color, lw=lw, label=name)
ax.set_title('Reliability Diagram — Calibración de Modelos', fontweight='bold')
ax.set_xlabel('Probabilidad Predicha (media por bin)')
ax.set_ylabel('Fracción de Positivos Reales')
ax.legend(fontsize=8)

ax2 = axes[1]
for (name, (_, color)), proba in zip(MODELS.items(), PROBAS.values()):
    ax2.hist(proba[y_test == 0], bins=40, alpha=0.3, color=color, density=True)
    ax2.hist(proba[y_test == 1], bins=40, alpha=0.7, color=color, density=True,
             histtype='step', lw=2, label=name)
ax2.set_title('Distribución Scores: Buenos (relleno) vs Malos (borde)', fontweight='bold', fontsize=10)
ax2.set_xlabel('PD Estimada'); ax2.set_ylabel('Densidad')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/sesion13_calibration.png', dpi=150)
plt.show()

---
## 💰 Sección 4 — Análisis Económico: Curvas de Ganancia, Lift y Cost-Benefit

Las **curvas de ganancia acumulada** y el **Lift** traducen la discriminación estadística
en impacto económico real:

> *¿Qué % de los morosos capturo si apruebo solo el X% más riesgoso?*

En banca se usan para definir **puntos de corte**, estimar **pérdidas evitadas**
y justificar el ROI del modelo frente a la dirección comercial.

> **Referencia:** Baesens & Van Gestel (2009, Cap. 5); Hand & Henley (1997), JRSS-A.

In [ ]:
# ── Curvas de Ganancia y Lift ────────────────────────────────────────────────
def cumulative_gains_lift(y_true, y_proba):
    df = pd.DataFrame({'y': y_true.values, 'p': y_proba})
    df = df.sort_values('p', ascending=False).reset_index(drop=True)
    df['pct_pop']     = (df.index + 1) / len(df)
    df['cum_bad_pct'] = df['y'].cumsum() / df['y'].sum()
    df['lift']        = df['cum_bad_pct'] / df['pct_pop']
    return df['pct_pop'].values, df['cum_bad_pct'].values, df['lift'].values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Modelo aleatorio')
ax.plot([0, y_test.mean(), 1], [0, 1, 1], 'gray', lw=1.2, ls=':', label='Modelo perfecto')
for (name, (_, color)), proba in zip(MODELS.items(), PROBAS.values()):
    pct, gain, _ = cumulative_gains_lift(y_test, proba)
    lw = 1.8 if 'Baseline' in name else 2.5
    ax.plot(pct, gain, color=color, lw=lw, label=name)
ax.axvline(0.3, color='gray', ls='--', lw=0.8, alpha=0.7)
ax.set_title('Curva de Ganancia Acumulada', fontsize=12, fontweight='bold')
ax.set_xlabel('% Población (score descendente)'); ax.set_ylabel('% Morosos Capturados')
ax.legend(fontsize=9)

ax2 = axes[1]
ax2.axhline(1.0, color='black', ls='--', lw=1.2, label='Sin modelo (Lift=1)')
for (name, (_, color)), proba in zip(MODELS.items(), PROBAS.values()):
    pct, _, lift = cumulative_gains_lift(y_test, proba)
    step = max(1, len(pct) // 100)
    lw = 1.8 if 'Baseline' in name else 2.5
    ax2.plot(pct[::step], lift[::step], color=color, lw=lw, label=name)
ax2.set_title('Curva de Lift', fontsize=12, fontweight='bold')
ax2.set_xlabel('% Población (score descendente)'); ax2.set_ylabel('Lift')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/sesion13_gain_lift.png', dpi=150)
plt.show()

In [ ]:
# ── Cost-Benefit Analysis ────────────────────────────────────────────────────
# Supuestos de negocio — ajusta según tu portafolio
LOAN_AMOUNT = 10_000   # Monto promedio del crédito
LGD         = 0.45     # Loss Given Default (Basilea tipico)
PROFIT_GOOD = 500      # Ganancia neta por crédito bueno aprobado
LOSS_BAD    = LOAN_AMOUNT * LGD

def expected_profit_curve(y_true, y_proba,
                           cutoffs=np.arange(0.01, 0.99, 0.01),
                           profit_good=PROFIT_GOOD, loss_bad=LOSS_BAD):
    profits = []
    for cut in cutoffs:
        # score >= cut → rechazamos (predecimos default=1)
        # score <  cut → aprobamos  (predecimos bueno=0)
        y_pred = (y_proba >= cut).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        # tn = buenos aprobados correctamente
        # fp = malos aprobados incorrectamente
        profit = tn * profit_good - fp * loss_bad
        profits.append(profit)
    return cutoffs, np.array(profits)

fig, ax = plt.subplots(figsize=(9, 5))
for (name, (_, color)), proba in zip(MODELS.items(), PROBAS.values()):
    cuts, profits = expected_profit_curve(y_test, proba)
    lw = 1.8 if 'Baseline' in name else 2.5
    ax.plot(cuts, profits / 1e6, color=color, lw=lw, label=name)

ax.axhline(0, color='black', ls='--', lw=0.8)
ax.set_title(f'Beneficio Esperado por Umbral de Corte\n'
             f'(LGD={LGD:.0%} | Ganancia por crédito bueno = ${PROFIT_GOOD:,})',
             fontweight='bold')
ax.set_xlabel('Umbral de Corte (Punto de Rechazo)')
ax.set_ylabel('Beneficio Total (Millones USD)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/sesion13_cost_benefit.png', dpi=150)
plt.show()

---
## 🔄 Sección 5 — Estabilidad y Robustez

Un buen AUC en test **no garantiza performance estable en el tiempo**.
La estabilidad es un requisito explícito en:
- Basilea II/III (BCBS, 2006): *"Models must perform consistently over time"*
- EBA Guidelines (2017): PSI < 0.10 para aprobación de modelo

| Indicador | Descripción | Umbral |
|-----------|-------------|--------|
| **CV Score (std)** | Varianza del AUC en validación cruzada | Std < 0.02 |
| **PSI** | Population Stability Index entre train y test | < 0.10 estable, < 0.25 monitorear |
| **Overfitting Ratio** | AUC_train / AUC_test | < 1.05 aceptable |

In [ ]:
# ── Validación Cruzada 5-Fold Estratificada ──────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_full = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)
y_full = pd.concat([y_train, y_test], axis=0).reset_index(drop=True)

cv_rows = []
for name, (model, _) in MODELS.items():
    res = cross_validate(model, X_full, y_full, cv=cv,
                         scoring='roc_auc', return_train_score=True)
    mean_test  = res['test_score'].mean()
    std_test   = res['test_score'].std()
    mean_train = res['train_score'].mean()
    overfit    = mean_train / (mean_test + 1e-9)
    cv_rows.append({
        'Modelo'           : name,
        'AUC CV (mean)'    : round(mean_test, 4),
        'AUC CV (std)'     : round(std_test, 4),
        'AUC Train'        : round(mean_train, 4),
        'Overfitting Ratio': round(overfit, 3),
        'Estable?'         : '✅' if std_test < 0.02 else '⚠️'
    })

cv_df = pd.DataFrame(cv_rows).set_index('Modelo')
print("\n🔄 Validación Cruzada 5-Fold Estratificada:\n")
print(cv_df.to_string())

In [ ]:
# ── Population Stability Index (PSI) ─────────────────────────────────────────
# Referencia: Siddiqi (2006); EBA Guidelines on IRB (2017)

def compute_psi(expected, actual, n_bins=10, epsilon=1e-4):
    """PSI entre distribución de scores en train (esperada) y test (actual)."""
    breaks = np.percentile(expected, np.linspace(0, 100, n_bins + 1))
    breaks[0]  -= epsilon
    breaks[-1] += epsilon
    e_cnt, _ = np.histogram(expected, bins=breaks)
    a_cnt, _ = np.histogram(actual,   bins=breaks)
    e_pct = np.maximum(e_cnt / len(expected), epsilon)
    a_pct = np.maximum(a_cnt / len(actual),   epsilon)
    return np.sum((a_pct - e_pct) * np.log(a_pct / e_pct))

psi_rows = []
for name, (model, _) in MODELS.items():
    train_sc = model.predict_proba(X_train)[:, 1]
    test_sc  = model.predict_proba(X_test)[:, 1]
    psi_val  = compute_psi(train_sc, test_sc)
    if psi_val < 0.10:
        flag = '✅ Estable'
    elif psi_val < 0.25:
        flag = '⚠️ Monitorear'
    else:
        flag = '❌ Inestable'
    psi_rows.append({'Modelo': name, 'PSI': round(psi_val, 5), 'Estado': flag})

psi_df = pd.DataFrame(psi_rows).set_index('Modelo')
print("\n📊 Population Stability Index (PSI) — Train vs Test:\n")
print(psi_df.to_string())

---
## 📋 Sección 6 — Dashboard de Comparación Consolidado

Consolidamos **todas las dimensiones** en un único panel de decisión,
siguiendo el framework multi-criterio propuesto en Lessmann et al. (2015).

In [ ]:
# ── Tabla Maestra ────────────────────────────────────────────────────────────
summary = pd.DataFrame(index=list(MODELS.keys()))

for name, proba in PROBAS.items():
    summary.loc[name, 'AUC-ROC'] = round(roc_auc_score(y_test, proba), 4)
    summary.loc[name, 'Gini']    = round(compute_gini(y_test, proba), 4)
    summary.loc[name, 'KS']      = round(compute_ks(y_test, proba), 4)
    summary.loc[name, 'Brier']   = round(brier_score_loss(y_test, proba), 5)

summary['AUC CV']      = cv_df['AUC CV (mean)'].values
summary['CV Std']      = cv_df['AUC CV (std)'].values
summary['PSI']         = psi_df['PSI'].values
summary['Overfitting'] = cv_df['Overfitting Ratio'].values

for col in ['AUC-ROC', 'Gini', 'KS', 'AUC CV']:
    summary[f'R_{col}'] = summary[col].rank(ascending=False).astype(int)
for col in ['Brier', 'CV Std', 'PSI', 'Overfitting']:
    summary[f'R_{col}'] = summary[col].rank(ascending=True).astype(int)

rank_cols = [c for c in summary.columns if c.startswith('R_')]
summary['Rank Promedio'] = summary[rank_cols].mean(axis=1).round(2)
summary = summary.sort_values('Rank Promedio')

main_cols = ['AUC-ROC','Gini','KS','Brier','AUC CV','CV Std','PSI','Overfitting','Rank Promedio']
print("\n🏆 TABLA MAESTRA (ordenada por Rank Promedio):\n")
print(summary[main_cols].to_string())

In [ ]:
# ── Radar Chart (Spider Plot) ─────────────────────────────────────────────────
radar_raw = pd.DataFrame({
    'Discriminacion (AUC)' : summary['AUC-ROC'],
    'Separacion (KS)'      : summary['KS'],
    'Calibracion (Brier)'  : summary['Brier'],
    'Estabilidad CV (std)' : summary['CV Std'],
    'Robustez (PSI)'       : summary['PSI'],
    'Sin Overfitting'      : summary['Overfitting'],
}, index=summary.index)

# Normalizar: columnas donde mayor = mejor
for col in ['Discriminacion (AUC)', 'Separacion (KS)']:
    col_min, col_max = radar_raw[col].min(), radar_raw[col].max()
    radar_raw[col] = (radar_raw[col] - col_min) / (col_max - col_min + 1e-9)

# Columnas donde menor = mejor → invertir
for col in ['Calibracion (Brier)', 'Estabilidad CV (std)', 'Robustez (PSI)', 'Sin Overfitting']:
    col_min, col_max = radar_raw[col].min(), radar_raw[col].max()
    radar_raw[col] = 1 - (radar_raw[col] - col_min) / (col_max - col_min + 1e-9)

categories = list(radar_raw.columns)
N = len(categories)
angles = [n / N * 2 * np.pi for n in range(N)] + [0]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

model_list = list(MODELS.keys())
for name, row in radar_raw.iterrows():
    color = PALETTE[model_list.index(name)]
    vals  = row.tolist() + [row.tolist()[0]]
    lw = 3 if 'Baseline' not in name else 1.8
    ls = '--' if 'Baseline' in name else '-'
    ax.plot(angles, vals, color=color, lw=lw, ls=ls, label=name)
    ax.fill(angles, vals, color=color, alpha=0.05)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=9)
ax.set_ylim(0, 1)
ax.set_title('Perfil Multi-Dimensional de Modelos\n(Mayor area = Mejor modelo)',
             fontsize=12, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.1), fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/sesion13_radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🏆 Sección 7 — Selección del Modelo Final

### Framework de Decisión Multi-Criterio

Siguiendo la práctica de bancos y reguladores (BIS BCBS Working Paper 14, 2005),
la selección considera **4 ejes ponderados**:

| Eje | Peso | Criterio |
|-----|------|----------|
| **Discriminación** | 35% | AUC-ROC / Gini / KS |
| **Calibración** | 30% | Brier Score + Hosmer-Lemeshow |
| **Estabilidad** | 20% | PSI + CV Std |
| **Robustez** | 15% | Overfitting Ratio |

> Para uso bajo IFRS 9 / Basilea, la calibración y estabilidad tienen peso regulatorio
> equivalente o superior a la discriminación pura (EBA, 2017; BCBS, 2005).

In [ ]:
# ── Score Compuesto Ponderado ─────────────────────────────────────────────────
def normalize(series, ascending=True):
    """Normaliza a [0,1]. ascending=True: mayor valor = mayor score."""
    s_min, s_max = series.min(), series.max()
    norm = (series - s_min) / (s_max - s_min + 1e-9)
    return norm if ascending else 1 - norm

score_df = pd.DataFrame(index=summary.index)
# Discriminación (35%)
score_df['s_auc']   = normalize(summary['AUC-ROC']) * 0.20
score_df['s_gini']  = normalize(summary['Gini'])    * 0.10
score_df['s_ks']    = normalize(summary['KS'])       * 0.05
# Calibración (30%)
score_df['s_brier'] = normalize(summary['Brier'], ascending=False) * 0.30
# Estabilidad (20%)
score_df['s_cvstd'] = normalize(summary['CV Std'], ascending=False) * 0.10
score_df['s_psi']   = normalize(summary['PSI'],    ascending=False) * 0.10
# Robustez (15%)
score_df['s_over']  = normalize(summary['Overfitting'], ascending=False) * 0.15

score_df['Score Final'] = score_df.sum(axis=1)
score_df = score_df.sort_values('Score Final', ascending=False)

print("\n🏆 RANKING FINAL POR SCORE COMPUESTO:\n")
display_cols = ['s_auc','s_gini','s_ks','s_brier','s_cvstd','s_psi','s_over','Score Final']
rename = {'s_auc':'AUC(20%)','s_gini':'Gini(10%)','s_ks':'KS(5%)',
          's_brier':'Calib(30%)','s_cvstd':'CVstd(10%)','s_psi':'PSI(10%)','s_over':'Overfit(15%)'}
print(score_df[display_cols].rename(columns=rename).to_string())

winner = score_df.index[0]
print(f"\n>>> MODELO SELECCIONADO: {winner} <<<")

In [ ]:
# ── Gráfica de Ranking Final ─────────────────────────────────────────────────
model_list = list(MODELS.keys())
fig, ax = plt.subplots(figsize=(9, 4))
colors_bar = [PALETTE[model_list.index(m)] for m in score_df.index[::-1]]
bars = ax.barh(score_df.index[::-1], score_df['Score Final'][::-1],
               color=colors_bar, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, score_df['Score Final'][::-1]):
    ax.text(val + 0.003, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=10, fontweight='bold')

ax.set_title('Score Compuesto de Seleccion de Modelo\n'
             '(Discriminacion 35% | Calibracion 30% | Estabilidad 20% | Robustez 15%)',
             fontweight='bold', fontsize=11)
ax.set_xlabel('Score Ponderado (mayor = mejor)')
ax.set_xlim(0, score_df['Score Final'].max() * 1.18)

winner_idx = list(score_df.index[::-1]).index(winner)
bars[winner_idx].set_edgecolor('gold')
bars[winner_idx].set_linewidth(3)

plt.tight_layout()
plt.savefig('../outputs/sesion13_ranking_final.png', dpi=150)
plt.show()

In [ ]:
# ── Guardar Modelo Final ─────────────────────────────────────────────────────
final_model = MODELS[winner][0]
joblib.dump(final_model, '../models/modelo_final_sesion13.pkl')

# Guardar PD estimadas → input para Sesión 15 (Calibración PD - IFRS 9)
pd_df = pd.DataFrame({'y_true': y_test.values, 'pd_estimada': PROBAS[winner]})
pd_df.to_csv('../data/pd_estimada_modelo_final.csv', index=False)

print(f"Modelo final guardado: {winner}")
print(f"  AUC-ROC : {summary.loc[winner,'AUC-ROC']:.4f}")
print(f"  Gini    : {summary.loc[winner,'Gini']:.4f}")
print(f"  KS      : {summary.loc[winner,'KS']:.4f}")
print(f"  PSI     : {summary.loc[winner,'PSI']:.5f}")
print("\nPD estimadas en '../data/pd_estimada_modelo_final.csv'")
print("→ Input directo para Sesion 15: Calibracion PD bajo IFRS 9")

---
## 📝 Sección 8 — Conclusiones y Recomendaciones

### Resumen de Hallazgos

| Dimensión | Mejor Modelo | Observación |
|-----------|-------------|-------------|
| Discriminación (AUC) | *(completar)* | *(completar)* |
| Calibración (Brier) | *(completar)* | *(completar)* |
| Estabilidad (PSI) | *(completar)* | *(completar)* |
| Score Compuesto | *(completar)* | **Modelo seleccionado** |

### Ganancia vs Baseline (Logistic Regression)

| Métrica | LR Baseline | Modelo Final | Δ Ganancia |
|---------|-------------|--------------|------------|
| AUC-ROC | *(completar)* | *(completar)* | *(completar)* |
| Gini    | *(completar)* | *(completar)* | *(completar)* |
| KS      | *(completar)* | *(completar)* | *(completar)* |

### Próximos Pasos

1. **Sesión 14 — Interpretabilidad:** SHAP values sobre el modelo final para explicar
   predicciones individuales (GDPR art. 22; SR 11-7 Fed Reserve).

2. **Sesión 15 — Calibración PD (IFRS 9):** Las PD estimadas del modelo final
   se usarán como insumo para calibración TTC vs PIT (EBA/GL/2017/16).

3. **Monitoreo en producción:** Alertas automáticas de PSI para detectar model drift
   (recalibrar si PSI > 0.10).

---

## 📚 Referencias Bibliográficas

1. **Siddiqi, N. (2006).** *Credit Risk Scorecards: Developing and Implementing Intelligent Credit Scoring.* Wiley.

2. **Thomas, L.C., Edelman, D.B., & Crook, J.N. (2002).** *Credit Scoring and Its Applications.* SIAM.

3. **Baesens, B. & Van Gestel, T. (2009).** *Credit Risk Management: Basic Concepts.* Oxford University Press.

4. **Lessmann, S., Baesens, B., Seow, H.V., & Thomas, L.C. (2015).** Benchmarking state-of-the-art classification algorithms for credit scoring: An update of research. *European Journal of Operational Research, 247*(1), 124-136.

5. **BIS Basel Committee on Banking Supervision (2005).** *Studies on the Validation of Internal Rating Systems.* BCBS Working Paper No. 14.

6. **EBA (2017).** *Guidelines on PD estimation, LGD estimation and the treatment of defaulted exposures.* EBA/GL/2017/16.

7. **Hand, D.J. & Henley, W.E. (1997).** Statistical classification methods in consumer credit scoring: a review. *Journal of the Royal Statistical Society: Series A, 160*(3), 523-541.

8. **Hosmer, D.W. & Lemeshow, S. (2000).** *Applied Logistic Regression* (2nd ed.). Wiley.

9. **Tasche, D. (2006).** *Validation of internal rating systems and PD estimates.* arXiv:physics/0606071.